In [2]:
!pip install imbalanced-learn


In [3]:
import pandas as pd
import numpy as np
import re
import nltk
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
# Load the dataset
df = pd.read_csv("spam.csv", encoding='latin-1')[['v1', 'v2']]


In [16]:
df

,label,message,cleaned,length
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...,111
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni,29
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...,155
3,0,U dun say so early hor... U c already then say...,u dun say early hor u c already say,49
4,0,"Nah I don't think he goes to usf, he lives aro...",nah dont think go usf life around though,61
...,...,...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...,nd time tried contact u u pound prize claim ea...,161
5568,0,Will Ì_ b going to esplanade fr home?,b going esplanade fr home,37
5569,0,"Pity, * was in mood for that. So...any other s...",pity mood soany suggestion,57
5570,0,The guy did some bitching but I acted like i'd...,guy bitching acted like id interested buying s...,125


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   v1      5572 non-null   object
 1   v2      5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [6]:
df.columns

Index(['v1', 'v2'], dtype='object')

In [7]:
df.describe

<bound method NDFrame.describe of         v1                                                 v2
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
...    ...                                                ...
5567  spam  This is the 2nd time we have tried 2 contact u...
5568   ham              Will Ì_ b going to esplanade fr home?
5569   ham  Pity, * was in mood for that. So...any other s...
5570   ham  The guy did some bitching but I acted like i'd...
5571   ham                         Rofl. Its true to its name

[5572 rows x 2 columns]>

In [8]:
df.columns = ['label', 'message']
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

df.head()


,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


Text Preprocessing (Lemmatization + Stopwords)

In [9]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(words)

df['cleaned'] = df['message'].apply(clean_text)
df['length'] = df['message'].apply(len)

df[['message', 'cleaned', 'length']].head()


,message,cleaned,length
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...,111
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni,29
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...,155
3,U dun say so early hor... U c already then say...,u dun say early hor u c already say,49
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think go usf life around though,61


Vectorization + Feature Engineering

In [10]:
vectorizer = TfidfVectorizer()
X_text = vectorizer.fit_transform(df['cleaned'])

# Combine TF-IDF with message length
from scipy.sparse import hstack
X = hstack([X_text, np.array(df['length']).reshape(-1, 1)])
y = df['label']


Handle Class Imbalance Using SMOTE

In [11]:
smote = SMOTE()
X_resampled, y_resampled = smote.fit_resample(X, y)


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42)


Train Logistic Regression with GridSearchCV

In [13]:
params = {'C': [0.1, 1, 10]}
log_model = GridSearchCV(LogisticRegression(max_iter=1000), params, cv=5)
log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)
print("🔍 Logistic Regression Classification Report:\n")
print(classification_report(y_test, y_pred))


🔍 Logistic Regression Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.99      0.99       985
           1       0.99      1.00      0.99       945

    accuracy                           0.99      1930
   macro avg       0.99      0.99      0.99      1930
weighted avg       0.99      0.99      0.99      1930



Compare with Naive Bayes


In [14]:
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("🔍 Naive Bayes Classification Report:\n")
print(classification_report(y_test, y_pred_nb))


🔍 Naive Bayes Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.91      0.95       985
           1       0.91      1.00      0.95       945

    accuracy                           0.95      1930
   macro avg       0.95      0.95      0.95      1930
weighted avg       0.96      0.95      0.95      1930



In [15]:
with open("spam_model.pkl", "wb") as f:
    pickle.dump(log_model, f)

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("✅ Model and vectorizer saved!")


✅ Model and vectorizer saved!
